In [3]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "amazonhelp_conversations.csv"
)

amazon_data = pd.read_csv(DATA_PATH)

print(amazon_data.shape)

amazon_data.head()

(168814, 5)


,customer_tweet_id,customer_text,brand_tweet_id,brand_text,brand_name
0,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,AmazonHelp
1,274,@AmazonHelp こちらこそありがとうございました。,275,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,AmazonHelp
2,272,amazonのfireTVstickが見れない😢,269,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,AmazonHelp
3,325,amazonプライムビデオ、再生エラーが多いです,324,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,AmazonHelp
4,616,@AmazonHelp 3 different people have given 3 di...,618,@115820 We'd like to take a further look into ...,AmazonHelp


In [2]:
amazon_data = pd.read_csv(DATA_PATH)

print("Shape:", amazon_data.shape)

amazon_data.head()

Shape: (168814, 5)


,customer_tweet_id,customer_text,brand_tweet_id,brand_text,brand_name
0,271,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,AmazonHelp
1,274,@AmazonHelp こちらこそありがとうございました。,275,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,AmazonHelp
2,272,amazonのfireTVstickが見れない😢,269,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,AmazonHelp
3,325,amazonプライムビデオ、再生エラーが多いです,324,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,AmazonHelp
4,616,@AmazonHelp 3 different people have given 3 di...,618,@115820 We'd like to take a further look into ...,AmazonHelp


In [4]:
import pandas as pd
import re

# Make a copy
intent_data = amazon_data.copy()

# Keep only useful columns
intent_data = intent_data[
    ["customer_text", "brand_text"]
].dropna()

# Remove duplicate customer messages
intent_data = intent_data.drop_duplicates(
    subset=["customer_text"]
)

# Basic text cleaning
def clean_text(text):
    text = str(text)
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)
    
    # Remove Twitter mentions
    text = re.sub(r"@\w+", " ", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()


intent_data["customer_text_clean"] = (
    intent_data["customer_text"]
    .apply(clean_text)
)

# Remove extremely short messages
intent_data = intent_data[
    intent_data["customer_text_clean"].str.len() >= 15
]

# Take a reproducible sample for intent discovery
sample_size = min(30_000, len(intent_data))

intent_sample = intent_data.sample(
    n=sample_size,
    random_state=42
).reset_index(drop=True)

print("Original conversations:", len(amazon_data))
print("After cleaning:", len(intent_data))
print("Intent discovery sample:", len(intent_sample))

print("\nSample messages:")
print(intent_sample["customer_text_clean"].head(10).to_string(index=False))

Original conversations: 168814
After cleaning: 147991
Intent discovery sample: 30000

Sample messages:
Contraseña incorrecta y ya la cambié 2 veces, a...
Hi, I’ve received a parcel today, signed for by...
why are so many in stock Prime delivery items s...
Well here we go yet again - PRIME item ordered ...
                   Required Invoice copy of below.
They didn’t advise anything they put me on hold...
latest iOS update on iPad changed orientation t...
x2 Parcels confirmed delivered to Amazon. 1 sel...
                                   Non pas encore.
I bought laptop on 04.10.17 but it’s showing fr...


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np

# TF-IDF representation
vectorizer = TfidfVectorizer(
    stop_words="english",
    lowercase=True,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.90,
    max_features=8000
)

X = vectorizer.fit_transform(
    intent_sample["customer_text_clean"]
)

print("TF-IDF shape:", X.shape)

TF-IDF shape: (30000, 8000)


In [6]:
# Discover initial customer-problem groups

N_CLUSTERS = 12

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    n_init=10
)

intent_sample["cluster"] = kmeans.fit_predict(X)

print("Clusters created:", N_CLUSTERS)
print("\nMessages per cluster:")

print(
    intent_sample["cluster"]
    .value_counts()
    .sort_index()
)

Clusters created: 12

Messages per cluster:
cluster
0       909
1      2203
2       833
3      1965
4      1762
5      1230
6      1495
7      2780
8       929
9      1705
10      912
11    13277
Name: count, dtype: int64


In [7]:
# Show the most important terms in every cluster

terms = np.array(
    vectorizer.get_feature_names_out()
)

order_centroids = (
    kmeans.cluster_centers_
    .argsort()[:, ::-1]
)

for cluster_id in range(N_CLUSTERS):

    top_terms = terms[
        order_centroids[cluster_id, :15]
    ]

    cluster_size = (
        intent_sample["cluster"] == cluster_id
    ).sum()

    print("\n" + "=" * 80)
    print(f"CLUSTER {cluster_id} | Messages: {cluster_size}")
    print("Top terms:", ", ".join(top_terms))


CLUSTER 0 | Messages: 909
Top terms: days, prime, order, days ago, ago, 10 days, delivery, amazon, 10, wait, refund, 15 days, package, 15, time

CLUSTER 1 | Messages: 2203
Top terms: amazon, amazon prime, prime, pay, amazon pay, india, account, package, amazon logistics, logistics, don, amazon india, just, time, balance

CLUSTER 2 | Messages: 833
Top terms: product, product delivered, delivered, received, return, refund, amazon, order, want, got, ordered, deliver, deliver product, time, delivery

CLUSTER 3 | Messages: 1965
Top terms: order, amazon, cancelled, placed, cancel, refund, help, cancel order, received, number, cancelled order, 408, id, order number, order id

CLUSTER 4 | Messages: 1762
Top terms: delivery, date, delivery date, today, amazon, prime, time, package, service, guaranteed, says, guaranteed delivery, delivery service, order, just

CLUSTER 5 | Messages: 1230
Top terms: delivered, package, today, says, delivered today, order, says delivered, package delivered, parcel

In [8]:
# Inspect real customer messages from each cluster

for cluster_id in sorted(intent_sample["cluster"].unique()):

    print("\n" + "=" * 100)
    print(f"CLUSTER {cluster_id}")

    examples = (
        intent_sample[
            intent_sample["cluster"] == cluster_id
        ]["customer_text_clean"]
        .head(5)
    )

    for i, text in enumerate(examples, 1):
        print(f"{i}. {text}")


CLUSTER 0
1. Well here we go yet again - PRIME item ordered 11/13 never arrived. Just checked &amp; now it says arriving Wed. So now Prime is 2 days, oh 3, forget it now it can be a week. Sorry to see I can no longer rely on amazon, cannot buy gifts.
2. has ruined my day, ruined my gift, and ruined my mood, because Amazon doesn't know how to count days for shipping, just your cash.
3. Hey my account has been locked for a few days with no reason and now I can't even login to contact support nor place an order.
4. I have already replied to the mail two days back and got no reply. Don't know what investigation is going on at your side.
5. Ur Amazon team till now they didn't refund my payment after 20 days also do solve my issue ✉ __email__ 📱7667342264

CLUSTER 1
1. I have ordered canon1300d dslr camera from amazon and they sent me rubbish broken speakers inside the box.
2. Have never experienced such an issue with Amazon. Amazon has infact delivered the products before schedule date. #pr

In [9]:
# Check common support keywords in customer messages

keywords = {
    "delivery": [
        "delivery", "deliver", "delivered", "package",
        "parcel", "shipping", "arrive", "arrived", "late"
    ],
    "refund": [
        "refund", "refunded", "money back", "reimburse"
    ],
    "cancellation": [
        "cancel", "cancelled", "canceled"
    ],
    "payment": [
        "payment", "charged", "charge", "billing",
        "card", "pay", "paid"
    ],
    "account": [
        "account", "login", "locked", "password",
        "sign in", "access"
    ],
    "return": [
        "return", "replacement", "replace"
    ],
    "prime": [
        "prime", "membership", "subscription"
    ],
    "product": [
        "product", "item", "broken", "damaged",
        "wrong item", "defective"
    ]
}

topic_counts = {}

texts = intent_sample["customer_text_clean"].str.lower()

for topic, words in keywords.items():

    pattern = "|".join(
        re.escape(word)
        for word in words
    )

    topic_counts[topic] = texts.str.contains(
        pattern,
        regex=True,
        na=False
    ).sum()

topic_counts_df = (
    pd.DataFrame(
        list(topic_counts.items()),
        columns=["topic", "messages"]
    )
    .sort_values(
        "messages",
        ascending=False
    )
    .reset_index(drop=True)
)

topic_counts_df

,topic,messages
0,delivery,7424
1,prime,2764
2,product,2639
3,payment,2149
4,account,1314
5,refund,1208
6,return,1065
7,cancellation,828


In [10]:
# Inspect real examples for each discovered topic

for topic, words in keywords.items():

    pattern = "|".join(
        re.escape(word)
        for word in words
    )

    mask = intent_sample["customer_text_clean"].str.lower().str.contains(
        pattern,
        regex=True,
        na=False
    )

    examples = (
        intent_sample.loc[
            mask,
            "customer_text_clean"
        ]
        .drop_duplicates()
        .head(10)
    )

    print("\n" + "=" * 100)
    print(f"TOPIC: {topic.upper()}")
    print(f"Matching messages: {mask.sum()}")

    for i, text in enumerate(examples, 1):
        print(f"{i}. {text}")


TOPIC: DELIVERY
Matching messages: 7424
1. Hi, I’ve received a parcel today, signed for by a neighbour who I don’t know, the box is like this. Items missing, can you help?
2. why are so many in stock Prime delivery items showing as “get it by Wednesday”. That’s three day delivery not next day delivery.
3. Well here we go yet again - PRIME item ordered 11/13 never arrived. Just checked &amp; now it says arriving Wed. So now Prime is 2 days, oh 3, forget it now it can be a week. Sorry to see I can no longer rely on amazon, cannot buy gifts.
4. latest iOS update on iPad changed orientation to portrait. Pls fix
5. x2 Parcels confirmed delivered to Amazon. 1 seller has received theirs, the other is saying they haven't. Need to track the parcels.. thanks
6. And while we're at out, my wife ordered something, paid for next day (to arrive yesterday) and that did not arrive either. So a lot of unhappy customers in this house right now.
7. cod ww2 comes out tomorrow. My copy still hasn’t dispatc